In [ ]:


!pip install -U spacy==3.6.1
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

In [ ]:
import pandas as pd
import re
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))


? Step 1: Import and merge all the three datasets.

In [ ]:
# loading commoncrawl datasets for both german and english
with open('/content/drive/MyDrive/commoncrawl_de_en.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

commoncrawl_de = pd.DataFrame(data, columns=['german'])

display("The shape of commoncrawl_de_en dataset : ", commoncrawl_de.shape)


with open('/content/drive/MyDrive/commoncrawl_en_de.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

commoncrawl_en = pd.DataFrame(data, columns=['english'])

display("The shape of commoncrawl_en_de dataset : ", commoncrawl_en.shape)


In [ ]:
# loading europarl datasets for both german and english

with open('/content/drive/MyDrive/europarl-v7_de_en.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

europarl_de = pd.DataFrame(data, columns=['german'])

display("The shape of europarl-v7_de_en dataset : ", europarl_de.shape)


with open('/content/drive/MyDrive/europarl-v7_en_de.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

europarl_en = pd.DataFrame(data, columns=['english'])

display("The shape of europarl-v7_en_de dataset : ", europarl_en.shape)


In [ ]:
# loading news commentary for both german and english

with open('/content/drive/MyDrive/news-commentary-v9_de_en.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

news_de = pd.DataFrame(data, columns=['german'])

display("The shape of news-commentary-v9_de_en dataset : ", news_de.shape)


with open('/content/drive/MyDrive/news-commentary-v9_en_de.txt', 'r', encoding='utf-8') as file:
    data = [line.strip() for line in file]

news_en = pd.DataFrame(data, columns=['english'])

display("The shape of news-commentary-v9_en_de dataset : ", news_en.shape)

Here we could see there some additional rows for the news commentary english dataset

In [ ]:
# Calculate individual sums as before
commoncrawl_de_rows = commoncrawl_de['german'].notna().sum()
commoncrawl_en_rows = commoncrawl_en['english'].notna().sum()
europarl_de_rows = europarl_de['german'].notna().sum()
europarl_en_rows = europarl_en['english'].notna().sum()
news_de_rows = news_de['german'].notna().sum()
news_en_rows = news_en['english'].notna().sum()

# Calculate total sums
total_de_rows = commoncrawl_de_rows + europarl_de_rows + news_de_rows
total_en_rows = commoncrawl_en_rows + europarl_en_rows + news_en_rows

# Print the total sums
print(f"Total German rows before merging: {total_de_rows}")
print(f"Total English rows before merging: {total_en_rows}")

Step 2: Data cleansing

In [ ]:
def load_and_clean_data(de_file_path, en_file_path, clean=False):
    """Loads and optionally cleans data from two files (German and English)."""

    def is_special_characters(line):
        return re.match(r'^[^a-zA-Z0-9\s]+$', line) is not None

    def clean_lines(file_path):
        cleaned_data = []
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                cleaned_line = line.strip()
                if cleaned_line and not cleaned_line.isdigit() and not is_special_characters(cleaned_line):
                    cleaned_line = re.sub(r'\s+', ' ', cleaned_line)
                    cleaned_data.append(cleaned_line)
        return cleaned_data

    de_data = clean_lines(de_file_path) if clean else [line.strip() for line in open(de_file_path, 'r', encoding='utf-8')]
    en_data = clean_lines(en_file_path) if clean else [line.strip() for line in open(en_file_path, 'r', encoding='utf-8')]

    de_df = pd.DataFrame(de_data, columns=['german'])
    en_df = pd.DataFrame(en_data, columns=['english'])

    return de_df.join(en_df)

# Load and process the datasets
commoncrawl = load_and_clean_data('/content/drive/MyDrive/commoncrawl_de_en.txt', '/content/drive/MyDrive/commoncrawl_en_de.txt')
europarl = load_and_clean_data('/content/drive/MyDrive/europarl-v7_de_en.txt', '/content/drive/MyDrive/europarl-v7_en_de.txt')
news = load_and_clean_data('/content/drive/MyDrive/news-commentary-v9_de_en.txt', '/content/drive/MyDrive/news-commentary-v9_en_de.txt', clean=True)

# Concatenate them into a single DataFrame
merged_df = pd.concat([commoncrawl, europarl, news], ignore_index=True)

display(merged_df)

In [ ]:
# Calculate individual sums as after merging
commoncrawl_de_rows = commoncrawl['german'].notna().sum()
commoncrawl_en_rows = commoncrawl['english'].notna().sum()
europarl_de_rows = europarl['german'].notna().sum()
europarl_en_rows = europarl['english'].notna().sum()
news_de_rows = news['german'].notna().sum()
news_en_rows = news['english'].notna().sum()

# Calculate total sums
total_de_rows = commoncrawl_de_rows + europarl_de_rows + news_de_rows
total_en_rows = commoncrawl_en_rows + europarl_en_rows + news_en_rows

# Print the total sums
print(f"Total German rows before merging: {total_de_rows}")
print(f"Total English rows before merging: {total_en_rows}")

? Step 3: NLP pre processing - Dataset suitable to be used for AIML model learning

After Cleaning the data by removing extra spaces or numbers as newlines we were able to make the row count same

In [ ]:
def clean_text(text):
  text = text.lower()
  text = re.sub(r'[^\w\s]', '', text)
  text = re.sub(r'\d+', '', text)
  text = re.sub(r'\s+', ' ', text)
  return text

merged_df['german'] = merged_df['german'].apply(clean_text)
merged_df['english'] = merged_df['english'].apply(clean_text)

In [ ]:
display(merged_df)

In [ ]:
merged_df.to_parquet('/content/drive/MyDrive/merged_df.parquet', index=False)


In [ ]:
import nltk
# Download the 'punkt_tab' resource
nltk.download('punkt_tab')


In [ ]:
# Install necessary libraries
!pip install transformers tokenizers nltk pyarrow


In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer
from tqdm import tqdm
from nltk.corpus import stopwords
import nltk
from multiprocessing import Pool

# Download stopwords
nltk.download("stopwords")
german_stopwords = set(stopwords.words("german"))
english_stopwords = set(stopwords.words("english"))

# Load the Parquet file
file_path = "/content/drive/MyDrive/merged_df.parquet"  # Replace with your file path
data = pd.read_parquet(file_path)

# def preprocess_text(text, language):
#     text = text.lower()  # Lowercase
#     stopwords_set = english_stopwords if language == "english" else german_stopwords
#     text = " ".join(word for word in text.split() if word not in stopwords_set)  # Remove stopwords
#     return text

# # Wrapper function for German
# def preprocess_german(text):
#     return preprocess_text(text, language="german")

# # Wrapper function for English
# def preprocess_english(text):
#     return preprocess_text(text, language="english")

# # Parallelize preprocessing
# def parallel_apply(func, data, workers=4):
#     with Pool(workers) as p:
#         return list(tqdm(p.imap(func, data), total=len(data)))

# # Apply preprocessing in parallel
# data['german'] = parallel_apply(preprocess_german, data['german'], workers=4)
# data['english'] = parallel_apply(preprocess_english, data['english'], workers=4)

# # Initialize tokenizers
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Using device: {device}")

# german_tokenizer = AutoTokenizer.from_pretrained("bert-base-german-cased")
# english_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# # Tokenize in batches with GPU
# def tokenize_column_with_gpu(texts, tokenizer, batch_size=10000):
#     tokenized_output = []
#     for batch in tqdm(range(0, len(texts), batch_size)):
#         batch_texts = texts[batch:batch + batch_size].tolist()
#         # Move tensors to GPU for tokenization
#         tokens = tokenizer(
#             batch_texts,
#             padding=True,
#             truncation=True,
#             return_tensors="pt"
#         ).input_ids.to(device)  # Send to GPU
#         tokenized_output.extend(tokens.cpu().tolist())  # Move back to CPU for storage
#     return tokenized_output

# # Tokenize both columns
# data['german_tokens'] = tokenize_column_with_gpu(data['german'], german_tokenizer)
# data['english_tokens'] = tokenize_column_with_gpu(data['english'], english_tokenizer)

# # Save the tokenized data back to Parquet
# data.to_parquet("tokenized_output_with_gpu.parquet", index=False)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

german_tokenizer = AutoTokenizer.from_pretrained("bert-base-german-cased", use_fast=True)
english_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", use_fast=True)

# Tokenization function
def fast_tokenize_column(texts, tokenizer, batch_size=50000):
    tokenized_output = []
    for start in tqdm(range(0, len(texts), batch_size), desc="Tokenizing"):
        batch_texts = texts[start:start + batch_size].tolist()
        tokens = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).input_ids.to(device)
        tokenized_output.extend(tokens.cpu().tolist())
    return tokenized_output

# Tokenize columns
data['german_tokens'] = fast_tokenize_column(data['german'], german_tokenizer)
data['english_tokens'] = fast_tokenize_column(data['english'], english_tokenizer)

# Save the results
output_path = "/content/drive/MyDrive/fast_tokenized_output.parquet"
data.to_parquet(output_path, index=False)
print(f"Tokenized data saved to {output_path}")


In [ ]:
# import pandas as pd
# data = pd.read_parquet("/content/drive/MyDrive/fast_tokenized_output.parquet")

# # Inspect the first few rows
# print(data.head())

In [ ]:
import dask.dataframe as dd

# Load the Parquet file as a Dask DataFrame
data = dd.read_parquet("/content/drive/MyDrive/fast_tokenized_output.parquet")

# Inspect the first few rows
print(data.head())

In [ ]:
print(data.tail())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
import dask.array as da

german_tokens_dask = data['german_tokens'].values
english_tokens_dask = data['english_tokens'].values

# 2. Compute the dask arrays to get numpy arrays.
german_tokens = da.compute(german_tokens_dask)
english_tokens = da.compute(english_tokens_dask)

max_length = 512  # Adjust as needed
german_tokens = pad_sequences(german_tokens, maxlen=max_length, padding='post', dtype='object')
english_tokens = pad_sequences(english_tokens, maxlen=max_length, padding='post', dtype='object')


# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(german_tokens, english_tokens, test_size=0.2, random_state=42)

# Convert to TensorFlow tensors with appropriate dtype
X_train = tf.convert_to_tensor(X_train, dtype=tf.int32)
X_val = tf.convert_to_tensor(X_val, dtype=tf.int32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)
y_val = tf.convert_to_tensor(y_val, dtype=tf.int32)

# Check GPU availability and explicitly assign device
device_name = tf.test.gpu_device_name()
if device_name:
    print(f"Found GPU at: {device_name}")
else:
    print("No GPU found, using CPU instead.")

# Define model parameters (adjust as needed)
embedding_dim = 64
rnn_units = 128
vocab_size = 30522

# Build and compile the model within the device scope
with tf.device(device_name):
    model = tf.keras.models.Sequential([
        tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_length),
        tf.keras.layers.SimpleRNN(rnn_units, return_sequences=False),
        tf.keras.layers.Dense(vocab_size, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Train and evaluate within the device scope
with tf.device(device_name):
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        batch_size=64,  # Adjust as needed
        epochs=5       # Adjust as needed
    )